# Credit Risk Analytics & Risk Modelling

## Real-world dataset edition

This notebook uses the **UCI Default of Credit Card Clients** dataset (dataset ID 350), with 30,000 historical records from Taiwan and a next-month default outcome. Source: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/350/defaultofcreditcardclients), DOI [10.24432/C55S3H](https://doi.org/10.24432/C55S3H).

**Responsible use:** This is a retrospective educational analysis of historical data from 2005. It is not a validated credit decision model and must not be used to approve/deny credit or make other consequential decisions. See `docs/ethics_and_limitations.md`.

## 1. Setup and load data

The data is retrieved from UCI at runtime, so an internet connection is required. No raw customer-level dataset is committed to this repository.

In [ ]:
%pip -q install ucimlrepo

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, ConfusionMatrixDisplay, roc_auc_score,
    roc_curve, balanced_accuracy_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TARGET = "default_next_month"
VISUALS_DIR = Path("../visualizations")
if not VISUALS_DIR.exists():
    VISUALS_DIR = Path("visualizations")
VISUALS_DIR.mkdir(parents=True, exist_ok=True)

dataset = fetch_ucirepo(id=350)
X_source = dataset.data.features.copy()
y_source = dataset.data.targets.copy()
X_source.columns = [str(c).strip() for c in X_source.columns]
y_source.columns = [str(c).strip() for c in y_source.columns]

df = X_source.copy()
df[TARGET] = y_source.iloc[:, 0].reset_index(drop=True)
aliases = {
    "X1":"LIMIT_BAL", "X2":"SEX", "X3":"EDUCATION", "X4":"MARRIAGE",
    "X5":"AGE", "X6":"PAY_0", "X7":"PAY_2", "X8":"PAY_3", "X9":"PAY_4",
    "X10":"PAY_5", "X11":"PAY_6", "X12":"BILL_AMT1", "X13":"BILL_AMT2",
    "X14":"BILL_AMT3", "X15":"BILL_AMT4", "X16":"BILL_AMT5",
    "X17":"BILL_AMT6", "X18":"PAY_AMT1", "X19":"PAY_AMT2", "X20":"PAY_AMT3",
    "X21":"PAY_AMT4", "X22":"PAY_AMT5", "X23":"PAY_AMT6"
}
df = df.rename(columns=aliases)
df[TARGET] = pd.to_numeric(df[TARGET], errors="raise").astype(int)
print("Dataset shape:", df.shape)
df.head()

## 2. Data quality checks

In [ ]:
print("Duplicate rows:", int(df.duplicated().sum()))
print("\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
display(df.describe(include="all").T)

## 3. Target distribution

Class imbalance matters: accuracy by itself can hide poor detection of the positive class.

In [ ]:
target_counts = df[TARGET].value_counts().sort_index()
target_rate = df[TARGET].mean()
ax = target_counts.plot(kind="bar", figsize=(7, 4), color="steelblue")
ax.set(title="Next-month default outcome (historical UCI data)",
       xlabel="Default outcome (0 = no, 1 = yes)", ylabel="Number of records")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig(VISUALS_DIR / "target_distribution_real_data.png", dpi=160, bbox_inches="tight")
plt.show()
print(f"Observed positive-class share in this dataset: {target_rate:.1%}")

## 4. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(df["LIMIT_BAL"], bins=35, edgecolor="white")
axes[0].set(title="Credit limit distribution", xlabel="Credit limit (NT dollars)", ylabel="Count")
axes[1].hist(df["AGE"], bins=25, edgecolor="white")
axes[1].set(title="Age distribution", xlabel="Age (years)", ylabel="Count")
pay_col = "PAY_0" if "PAY_0" in df.columns else "X6"
df[pay_col].value_counts().sort_index().plot(kind="bar", ax=axes[2], color="darkcyan")
axes[2].set(title="September 2005 repayment status", xlabel="Repayment status code", ylabel="Count")
axes[2].tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(VISUALS_DIR / "feature_distributions_real_data.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
numeric = df.select_dtypes(include="number").drop(columns=["ID"], errors="ignore")
corr = numeric.corr()
fig, ax = plt.subplots(figsize=(12, 9))
image = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr.columns)), labels=corr.columns, rotation=90)
ax.set_yticks(range(len(corr.columns)), labels=corr.columns)
fig.colorbar(image, ax=ax, label="Pearson correlation")
ax.set_title("Numeric-variable correlation matrix")
fig.tight_layout()
fig.savefig(VISUALS_DIR / "correlation_heatmap_real_data.png", dpi=160, bbox_inches="tight")
plt.show()

## 5. Baseline modelling

The ID field is excluded. Coded demographic and repayment-status fields are treated as categorical in this baseline to avoid assuming equal numeric spacing between codes. All preprocessing is fitted inside the training pipeline.

In [ ]:
categorical_columns = [
    "SEX", "EDUCATION", "MARRIAGE",
    "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"
]
X = df.drop(columns=[TARGET, "ID"], errors="ignore")
y = df[TARGET]
categorical = [c for c in categorical_columns if c in X.columns]
numeric = [c for c in X.columns if c not in categorical]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric),
    ("categorical", categorical_pipeline, categorical)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    )
}
print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("Categorical features:", categorical)
print("Numeric features:", numeric)

## 6. Evaluation

Review precision and recall for each class as well as ROC-AUC and balanced accuracy. These metrics do not establish business utility, fairness, calibration, or suitability for deployment.

In [ ]:
results = []
fig_roc, ax_roc = plt.subplots(figsize=(7, 5))

for name, estimator in models.items():
    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", estimator)
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, prob)
    bal_acc = balanced_accuracy_score(y_test, pred)
    print(f"\n=== {name} ===")
    print(classification_report(y_test, pred, zero_division=0))
    print(f"ROC-AUC: {auc:.3f}")
    print(f"Balanced accuracy: {bal_acc:.3f}")
    results.append({
        "model": name,
        "roc_auc": auc,
        "balanced_accuracy": bal_acc
    })
    ConfusionMatrixDisplay.from_predictions(y_test, pred, values_format="d")
    plt.title(f"{name} — UCI historical data")
    plt.tight_layout()
    plt.savefig(VISUALS_DIR / f"confusion_matrix_{name.lower().replace(' ', '_')}_real_data.png",
                dpi=160, bbox_inches="tight")
    plt.show()
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax_roc.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

ax_roc.plot([0, 1], [0, 1], "--", label="Chance baseline")
ax_roc.set(title="ROC curves — UCI Default of Credit Card Clients",
           xlabel="False positive rate", ylabel="True positive rate")
ax_roc.legend()
fig_roc.tight_layout()
fig_roc.savefig(VISUALS_DIR / "roc_curves_real_data.png", dpi=160, bbox_inches="tight")
plt.show()
results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
display(results_df)

## 7. Interpretation checklist

Before writing conclusions, use the actual metrics from the executed notebook. Do not copy example scores from other runs.

- Which model has higher ROC-AUC on this fixed test split?
- How does positive-class recall compare with precision?
- What kinds of errors appear in each confusion matrix?
- Why can a model with higher accuracy still be less useful for detecting defaults?
- Which limitations prevent these results from being used for current credit decisions?

## 8. Limitations and citation

This is a single stratified holdout evaluation on historical Taiwanese credit-card data from 2005. It is not representative of current applicants by assumption, does not establish causal relationships, and has not been audited for fairness, calibrated for operational use, or validated out of time.

**Dataset citation:** Yeh, I. (2009). *Default of Credit Card Clients* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C55S3H. Licensed under CC BY 4.0.

See [dataset card](../docs/dataset_card.md), [methodology](../docs/methodology.md), [data dictionary](../docs/data_dictionary.md), and [ethics and limitations](../docs/ethics_and_limitations.md).